<a href="https://colab.research.google.com/github/DavidDevKing/extractive-summarization-system-for-medical-reports/blob/main/notebooks/training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone "https://github.com/DavidDevKing/extractive-summarization-system-for-medical-reports.git"
!pip install -r extractive-summarization-system-for-medical-reports/notebooks/requirements.txt

import sys
sys.path.append("extractive-summarization-system-for-medical-reports/notebooks/")
from utils import *



import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel, AdamW
from datasets import load_dataset
from rouge_score import rouge_scorer
import numpy as np

# 1. SETUP & DATA LOADING
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading 200 medical records...")
full_dataset = load_dataset("ccdv/pubmed-summarization", "section", split="train")
subset_dataset = full_dataset.select(range(200))

# 2. GREEDY MATCHING LOGIC (Label Generation)
def get_extractive_labels(article_sents, abstract_text, top_k=3):
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    selected_indices = []

    for _ in range(top_k):
        best_rouge = -1
        best_idx = -1
        for i, sent in enumerate(article_sents):
            if i in selected_indices or len(sent.strip()) < 5: continue
            current_summary = " ".join([article_sents[idx] for idx in selected_indices] + [sent])
            score = scorer.score(abstract_text, current_summary)['rouge1'].fmeasure
            if score > best_rouge:
                best_rouge = score
                best_idx = i
        if best_idx != -1:
            selected_indices.append(best_idx)

    return [1 if i in selected_indices else 0 for i in range(len(article_sents))]

# 3. PREPROCESSING THE SUBSET
print("Preprocessing and Labeling...")
processed_data = []
for entry in subset_dataset:
    sentences = entry['article'].split(". ")[:20] # Limit to first 20 sents for memory
    labels = get_extractive_labels(sentences, entry['abstract'])

    for sent, label in zip(sentences, labels):
        encodings = tokenizer(sent, truncation=True, padding='max_length', max_length=128, return_tensors="pt")
        processed_data.append({
            'input_ids': encodings['input_ids'].flatten(),
            'attention_mask': encodings['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.float)
        })

# 4. MODEL ARCHITECTURE
class BioExtractor(nn.Module):
    def __init__(self, model_name):
        super(BioExtractor, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(768, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.sigmoid(self.classifier(cls_output))

model = BioExtractor(model_name).to(device)

# 5. THE TRAINING LOOP
train_loader = DataLoader(processed_data, batch_size=8, shuffle=True)
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCELoss()

print("Starting Training (Approx 15 mins)...")
for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device).view(-1, 1)

        outputs = model(ids, mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

# 6. SAVE THE TRAINED BRAIN
torch.save(model.state_dict(), "med_summarizer_trained.pt")
print("Training Complete. File 'med_summarizer_trained.pt' is ready.")

fatal: destination path 'extractive-summarization-system-for-medical-reports' already exists and is not an empty directory.
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl (12.8 MB)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

Generating validation split:   0%|          | 0/6633 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6658 [00:00<?, ? examples/s]

Preprocessing and Labeling...


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Starting Training (Approx 15 mins)...
Epoch 1 | Loss: 0.3833
Epoch 2 | Loss: 0.3092
Epoch 3 | Loss: 0.1849
Training Complete. File 'med_summarizer_trained.pt' is ready.
